# Context

Surge stock detector one of the setup for the kq-trending strategy

 find surge stock in given time horizon, for my trading, I should focus on any stock that has a big surge at daily and weekly level. 

In order to avoid manipulated stocks, I should exclude small cap stocks: this can be done by filter scanning symbols

The surge of stock is an alert meaning something changed on the underlying stock, it typically should coincidant with some news. So we still need some fundamental analysis

## Methodology

* Use trend_score_multi_horizon to normalize the  trending strength in given horizon for 3 different internval: daily, weekly, and monthly. looks 2 or 5 is a good start window
* if use trending_score based on its sign value like more than 0.5, then it might too late if the previous peak or valley is skewed super far away. 
* so we use a rolling min and max to detect if we move to the opposite direction quickly enough. 

## Super parameter optimization:

The moving average window for the trending_score_multi_horizon as well as 

Based on the change from last max or min is not good since the reverse signal is too strong. Probably just based on the moving average is good enough and is more robust. 

In [1]:
from sts.quant.indicators.trend import ma_trend
from sts.dio.equity import Ticker
from sts.dio.symbols import sp500_symbol_table
from sts.quant.candle import Candle
from sts.strategy.trending.kqtrend.surgestock import SurgeStock, SurgeStockMinMax
import pandas as pd
import numpy as np

In [2]:
ticker = Ticker("GLD")
df = ticker.history(period="10y")
df = Candle(df).log()

In [4]:
# test ma_trend
if False:
    trend_score_dict = ma_trend.trend_score_multi_horizon(df)
    pd.options.plotting.backend = "plotly"
    trend_score_dict["WeeklyScore"].plot()

In [7]:
surge_tock = SurgeStockMinMax(trend_score_window=5, trend_score_min_max_rolling_window=5, up_down_threshold=0.3)

In [8]:
surge_status = surge_tock(df)

In [9]:
pd.options.plotting.backend = "plotly"
surge_status["MonthlyScore"].plot()

In [10]:
df.resample("1ME").plot()